In [ ]:
import os
from shapely.geometry import Polygon
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.draw import polygon
import tifffile
import seaborn as sns

In [ ]:
# Global variables
metadata_list = [
'low_quality',
"POD",
"PT",
"CNT",
"DCT",
"TAL",
"DTL",
"ATL",
"IC",
"altPOD",
"altPT",
"altCNT",
"altDCT",
"altTAL",
"altDTL",
"altATL",
"altIC",
"PC",
"PEC",
"MD",
"EC",
"EC-GC",
"FIB",
"infFIB",
"pvFIB",
"aFIB",
"B",
"DC",
"MON",
"N",
"NK",
"PL",
"T",
"MAC",
"resMAC",
"moMAC",
"moMAC-INF",
"MAST",
"Lymphoid",
"Myeloid",
"NEU",
"SC/NEU",
"MC",
"MYOF",
"REN",
"VSM/P",
"unassigned"
]

### Step 1. Create Segmentation Image and CSV

Run to generate cell boundaries .csv with pixel coordinates as well as cell segmentation mask for registration (can also use H&E)

__file_path__ : .parquet file of cell boundaries

__metadata_path__ : .csv of cell ids and corresponding cell label

In [ ]:
# Load and process necessary files
coord_path = r"C:\Users\jvill\Downloads\ref_cell_boundaries.parquet"
metadata_path = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_KPMP_038_ref-selected\_metadata.csv"

coord_df = pd.read_parquet(coord_path)
metadata_df = pd.read_csv(metadata_path)
metadata_df.rename(columns={'cell_ID': 'cell_id'}, inplace=True)
output_dir = os.path.dirname(file_path)

In [ ]:
# Functions 
def coords_to_pix(df, col, padding=0):
    df_coord = df[col].values
    diff_coord = np.diff(df_coord)
    min_diff = np.min(diff_coord[diff_coord>0])

    diff_rounded = np.round(diff_coord/min_diff)
    first_coord = np.round(df[col].iloc[0])
    pix_coord = np.round((df_coord - df_coord.min()) / min_diff).astype(int) + padding

    return pix_coord, min_diff


def convert_xenium_to_pix_coords(coord_df, output_dir, metadata_df, metadata_list, coord_cols=['vertex_x', 'vertex_y'], label_col='label_id', padding=0, downsampling=1, output_name='ref'):
    print('Converting Xenium coordinates to pixel coordinates...')

    df = coord_df.copy()
    coord_x, min_x = coords_to_pix(df, coord_cols[0], padding)
    coord_y, min_y = coords_to_pix(df, coord_cols[1], padding)

    
    if downsampling > 1:
        print(f"Downsampling by {downsampling}")
        ds_x = coord_x / downsampling
        ds_y = coord_y / downsampling

        df['x'] = ds_x
        df['y'] = ds_y
        x_bounds = int(ds_x.max()) + 1
        y_bounds = int(ds_y.max()) + 1

        res_x = 1 / (float(min_x) * downsampling)
        res_y = 1 / (float(min_y) * downsampling)
    else:
        print("No downsampling")
        df['x'] = coord_x
        df['y'] = coord_y
        x_bounds = int(coord_x.max()) + 1
        y_bounds = int(coord_y.max()) + 1

        res_x = 1 / float(min_x)
        res_y = 1 / float(min_y)

    mask = np.zeros((y_bounds+(2*padding), x_bounds+(2*padding)), dtype=np.uint8)
    
    # Pre-calculate metadata mapping for fast O(1) lookup
    label_to_idx = {label: i for i, label in enumerate(metadata_list)}
    # Map cell_ID to label index directly
    cell_to_idx = metadata_df.set_index('cell_id')['levelKID_refined'].map(label_to_idx).to_dict()
    # Default index for "unassigned" if missing (assuming it's at the end or handle accordingly)
    if 'unassigned' in metadata_list:
        unassigned_idx = label_to_idx.get('unassigned', 0)

    for label_id, group in df.groupby(label_col):

        cell_id = group['cell_id'].iloc[0]
        cell_label_idx = cell_to_idx.get(cell_id, unassigned_idx)
        
        # Draw polygon
        r = group['y'].values
        c = group['x'].values

        rr, cc = polygon(r, c, shape=mask.shape)
        mask[rr, cc] = cell_label_idx + 1

    tif_name = os.path.join(output_dir, f'{output_name}_cell_segmentation.tif')
    tifffile.imwrite(
        tif_name,
        mask,
        imagej=True,
        resolution=(res_x, res_y),
        resolutionunit=1,
        metadata={
            'axes': 'YX',       
            'unit':  'micron',     # Physical unit string (e.g. 'um', 'micron')
            'spacing': 1,  
        },
    )
    df.to_csv(os.path.join(output_dir, f'{output_name}_cell_segmentation.csv'), index=False)


In [ ]:
# Create image mask and .csv
convert_xenium_to_pix_coords(coord_df, output_dir, metadata_df, metadata_list, output_name='ref-test')

### Step 2. Use image2image to register segmentation mask to pseudo-histology or other SRS image (can also register H&E to pseudo-histology)
1. Set segmentation mask as moving image
2. Set SRS image as fixed image
3. Set fiducials/landmarks at corresponding image locations

### Step 3. Use image2image transform .json to register parquet file vertices to SRS
Run to transform Xenium boundary coordinates for any image/roi into the image space 

__transform_json__ : .json file for affine transformation from image2image project

__cell_seg_csv__: .csv file generated from step 1 for segmented cells in pixel space

__metadata_path__ : .csv of cell ids and corresponding cell label

In [ ]:
# Load and process necessary files
transform_json = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3782_aki-selected\aki_cell_segmentation-roi2_transform.i2r.json"
cell_seg_csv = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3782_aki-selected\aki_cell_segmentation.csv"
cell_seg_df = pd.read_csv(cell_seg_csv)
if "Unnamed: 0" in cell_seg_df.columns:
    cell_seg_df = cell_seg_df.drop(columns=["Unnamed: 0"]) 
metadata_path= r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3782_aki-selected\_metadata.csv"
metadata_df = pd.read_csv(metadata_path)
metadata_df.rename(columns={'cell_ID': 'cell_id'}, inplace=True)

In [ ]:
# Functions 
from util_transform import CellDataMapper, apply_roi_transform_from_json
def transform_and_find_cells(transform_json, metadata_df, cell_seg_df, output_dir, output_name='ref-test'):


    img_cells_transformed, img_bounds = apply_roi_transform_from_json(
        transform_json, 
        cell_seg_df, 
        clip_to_roi=False, # Crop image to only preserve roi bounds (only needed for roi)
        return_roi_bounds=True,  
        use_inverse=False # Use inverse of transform if fixed and moving images are flipped
        )

    img_cells_transformed_cropped = img_cells_transformed[
        (img_cells_transformed['x_roi'] >= 0) & 
        (img_cells_transformed['y_roi'] >= 0) &
        (img_cells_transformed['x_roi'] < img_bounds['width']) &
        (img_cells_transformed['y_roi'] < img_bounds['height'])
    ].copy()

    # Find all cells in image (including those cropped by the image boundaries)
    cells_in_img = img_cells_transformed_cropped['cell_id'].unique()
    total_roi_cells = img_cells_transformed[img_cells_transformed['cell_id'].isin(cells_in_img)].copy()
    total_roi_cells[['x_roi', 'y_roi']] = total_roi_cells[['x_roi', 'y_roi']].clip(lower=[0, 0], upper=[img_bounds['width'], img_bounds['height']], axis=1)
    total_roi_cells = total_roi_cells.merge(metadata_df, on='cell_id', how='left')
    # Flip rois vertically (sometimes needed for correcting transform)
    total_roi_cells['y_roi'] = img_bounds['height'] - total_roi_cells['y_roi']
    total_roi_cells.to_parquet(os.path.join(output_dir, f'{output_name}_img_segmented_cells.parquet'), index=False)

In [ ]:
# Transform parquet for masking and final registration
transform_and_find_cells(transform_json, metadata_df, cell_seg_df, output_dir, output_name='aki-roi2-test')